<a href="https://colab.research.google.com/github/Sammystar434/medical-diagnosis-network/blob/model-tuning/Model_Improvement_and_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [129]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay)
from google.colab import files

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Load the processed data

In [130]:
# 1. LOAD THE PROCESSED DATA
# (the three split files from the preprocessing notebook)

base_url = "https://raw.githubusercontent.com/Group-22-AI-ML/medical-diagnosis-network/main/Model/Preprocessing/"

train = pd.read_csv(base_url + "train.csv")
val = pd.read_csv(base_url + "validation.csv")
test = pd.read_csv(base_url + "test.csv")

X_train = train.drop("class", axis=1)
y_train = train["class"].values

X_val = val.drop("class", axis=1)
y_val = val["class"].values

X_test = test.drop("class", axis=1)
y_test = test["class"].values

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Xtr, Xva, Xte = (d.values.astype("float32") for d in (X_train, X_val, X_test))

Train: (364, 10)
Validation: (78, 10)
Test: (78, 10)


# 5. Experimentation Function

In [131]:
experiment_results = []

In [132]:
def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_layers=[16, 8],
    dropout_rate=0.0,
    learning_rate=0.001,
    batch_size=32,
    patience=10,
    experiment_name="Experiment",
    seed=42
):

    # keras.utils.set_random_seed(seed)

    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))

    # First hidden layer
    model.add(layers.Dense(hidden_layers[0], activation="relu",
        )
    )
    if dropout_rate > 0:
        model.add(layers.Dropout(dropout_rate))

    # Add any additional hidden layers
    for units in hidden_layers[1:]:
        model.add(
            layers.Dense(
                units,
                activation="relu"
            )
        )

        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation="sigmoid"))

    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True,
        verbose=1
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=150,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )

    probabilities = model.predict(
        X_val,
        verbose=0
    )

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    precision = precision_score(
        y_val,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        predictions,
        zero_division=0
    )

    cm = confusion_matrix(
        y_val,
        predictions
    )

    TN, FP, FN, TP = cm.ravel()

    experiment_results.append({
        "Experiment": experiment_name,
        "Architecture": str(hidden_layers),
        "Dropout": dropout_rate,
        "Learning Rate": learning_rate,
        "Batch Size": batch_size,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Confusion Matrix": cm
    })

    print("Experiment:", experiment_name)
    print("Architecture:", hidden_layers)
    print("Dropout:", dropout_rate)
    print("Learning Rate:", learning_rate)
    print("Batch Size:", batch_size)

    print("\nValidation Results")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nConfusion Matrix")
    print(cm)

    print("\nConfusion Matrix Breakdown")
    print(f"True Negatives  (TN): {TN}")
    print(f"False Positives (FP): {FP}")
    print(f"False Negatives (FN): {FN}")
    print(f"True Positives  (TP): {TP}")

    # 8. Return everything we need later
    return {
        "model": model,
        "history": history,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm
    }

# 6. Architechture Tuning

6.1 Deeper Experiment + DropOut

In [133]:
dropout_01_result = run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_layers=[32, 16, 8],
    dropout_rate=0.1,
    learning_rate=0.001,
    batch_size=32,
    experiment_name="Dropout 0.1"
)

Epoch 34: early stopping
Restoring model weights from the end of the best epoch: 24.
Experiment: Dropout 0.1
Architecture: [32, 16, 8]
Dropout: 0.1
Learning Rate: 0.001
Batch Size: 32

Validation Results
Accuracy : 0.9103
Precision: 0.9773
Recall   : 0.8776
F1-score : 0.9247

Confusion Matrix
[[28  1]
 [ 6 43]]

Confusion Matrix Breakdown
True Negatives  (TN): 28
False Positives (FP): 1
False Negatives (FN): 6
True Positives  (TP): 43


In [134]:
deeper_dropout_result = run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_layers=[32, 16, 8],
    dropout_rate=0.2,
    learning_rate=0.001,
    batch_size=32,
    experiment_name="Dropout 0.2"
)

Epoch 36: early stopping
Restoring model weights from the end of the best epoch: 26.
Experiment: Dropout 0.2
Architecture: [32, 16, 8]
Dropout: 0.2
Learning Rate: 0.001
Batch Size: 32

Validation Results
Accuracy : 0.8974
Precision: 0.9362
Recall   : 0.8980
F1-score : 0.9167

Confusion Matrix
[[26  3]
 [ 5 44]]

Confusion Matrix Breakdown
True Negatives  (TN): 26
False Positives (FP): 3
False Negatives (FN): 5
True Positives  (TP): 44


In [135]:
deeper_dropout_03_result = run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_layers=[32, 16, 8],
    dropout_rate=0.3,
    learning_rate=0.001,
    batch_size=32,
    experiment_name="Dropout 0.3"
)

Epoch 56: early stopping
Restoring model weights from the end of the best epoch: 46.
Experiment: Dropout 0.3
Architecture: [32, 16, 8]
Dropout: 0.3
Learning Rate: 0.001
Batch Size: 32

Validation Results
Accuracy : 0.9359
Precision: 0.9783
Recall   : 0.9184
F1-score : 0.9474

Confusion Matrix
[[28  1]
 [ 4 45]]

Confusion Matrix Breakdown
True Negatives  (TN): 28
False Positives (FP): 1
False Negatives (FN): 4
True Positives  (TP): 45


In [136]:
lr_0005_result = run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_layers=[32, 16, 8],
    dropout_rate=0.2,
    learning_rate=0.0005,
    batch_size=32,
    experiment_name="LR 0.0005"
)

Epoch 64: early stopping
Restoring model weights from the end of the best epoch: 54.
Experiment: LR 0.0005
Architecture: [32, 16, 8]
Dropout: 0.2
Learning Rate: 0.0005
Batch Size: 32

Validation Results
Accuracy : 0.9103
Precision: 0.9565
Recall   : 0.8980
F1-score : 0.9263

Confusion Matrix
[[27  2]
 [ 5 44]]

Confusion Matrix Breakdown
True Negatives  (TN): 27
False Positives (FP): 2
False Negatives (FN): 5
True Positives  (TP): 44


In [137]:
lr_001_result = run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_layers=[32, 16, 8],
    dropout_rate=0.2,
    learning_rate=0.01,
    batch_size=32,
    experiment_name="LR 0.01"
)

Epoch 13: early stopping
Restoring model weights from the end of the best epoch: 3.
Experiment: LR 0.01
Architecture: [32, 16, 8]
Dropout: 0.2
Learning Rate: 0.01
Batch Size: 32

Validation Results
Accuracy : 0.9231
Precision: 0.9574
Recall   : 0.9184
F1-score : 0.9375

Confusion Matrix
[[27  2]
 [ 4 45]]

Confusion Matrix Breakdown
True Negatives  (TN): 27
False Positives (FP): 2
False Negatives (FN): 4
True Positives  (TP): 45


In [138]:
batch_16_result = run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_layers=[32, 16, 8],
    dropout_rate=0.2,
    learning_rate=0.001,
    batch_size=16,
    experiment_name="Batch-size 16"
)

Epoch 26: early stopping
Restoring model weights from the end of the best epoch: 16.
Experiment: Batch-size 16
Architecture: [32, 16, 8]
Dropout: 0.2
Learning Rate: 0.001
Batch Size: 16

Validation Results
Accuracy : 0.9231
Precision: 0.9778
Recall   : 0.8980
F1-score : 0.9362

Confusion Matrix
[[28  1]
 [ 5 44]]

Confusion Matrix Breakdown
True Negatives  (TN): 28
False Positives (FP): 1
False Negatives (FN): 5
True Positives  (TP): 44


In [139]:
batch_64_result = run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_layers=[32, 16, 8],
    dropout_rate=0.2,
    learning_rate=0.001,
    batch_size=64,
    experiment_name="Batch-size 64"
)

Epoch 56: early stopping
Restoring model weights from the end of the best epoch: 46.
Experiment: Batch-size 64
Architecture: [32, 16, 8]
Dropout: 0.2
Learning Rate: 0.001
Batch Size: 64

Validation Results
Accuracy : 0.9231
Precision: 0.9778
Recall   : 0.8980
F1-score : 0.9362

Confusion Matrix
[[28  1]
 [ 5 44]]

Confusion Matrix Breakdown
True Negatives  (TN): 28
False Positives (FP): 1
False Negatives (FN): 5
True Positives  (TP): 44


# Summary of Experiments

In [ ]:
experiment_results.clear()

In [140]:
if not experiment_results:
    experiment_vars = {
        "Dropout 0.1": "dropout_01_result" in globals() and dropout_01_result,
        "Dropout 0.2": "deeper_dropout_result" in globals() and deeper_dropout_result,
        "Dropout 0.3": "deeper_dropout_03_result" in globals() and deeper_dropout_03_result,
        "LR 0.0005": "lr_0005_result" in globals() and lr_0005_result,
        "LR 0.01": "lr_001_result" in globals() and lr_001_result,
        "Batch-size 16": "batch_16_result" in globals() and batch_16_result,
        "Batch-size 64": "batch_64_result" in globals() and batch_64_result,
    }
    for name, res in experiment_vars.items():
        if res:
            experiment_results.append({
                "Experiment": name,
                "Architecture": str(res.get("model").input_shape) if res.get("model") else "",
                "Dropout": res.get("dropout_rate", "Unknown"),
                "Learning Rate": res.get("learning_rate", "Unknown"),
                "Batch Size": res.get("batch_size", "Unknown"),
                "Accuracy": res.get("accuracy"),
                "Precision": res.get("precision"),
                "Recall": res.get("recall"),
                "F1 Score": res.get("f1"),
                "Confusion Matrix": res.get("confusion_matrix")
            })

experiment_summary = pd.DataFrame(experiment_results)

if not experiment_summary.empty and "F1 Score" in experiment_summary.columns:
    experiment_summary = experiment_summary.sort_values(
        by="F1 Score",
        ascending=False
    ).reset_index(drop=True)

experiment_summary

,Experiment,Architecture,Dropout,Learning Rate,Batch Size,Accuracy,Precision,Recall,F1 Score,Confusion Matrix
0,Dropout 0.3,"[32, 16, 8]",0.3,0.0010,32,0.935897,0.978261,0.918367,0.947368,"[[28, 1], [4, 45]]"
1,LR 0.01,"[32, 16, 8]",0.2,0.0100,32,0.923077,0.957447,0.918367,0.937500,"[[27, 2], [4, 45]]"
2,Batch-size 16,"[32, 16, 8]",0.2,0.0010,16,0.923077,0.977778,0.897959,0.936170,"[[28, 1], [5, 44]]"
3,Batch-size 64,"[32, 16, 8]",0.2,0.0010,64,0.923077,0.977778,0.897959,0.936170,"[[28, 1], [5, 44]]"
4,LR 0.0005,"[32, 16, 8]",0.2,0.0005,32,0.910256,0.956522,0.897959,0.926316,"[[27, 2], [5, 44]]"
5,Dropout 0.1,"[32, 16, 8]",0.1,0.0010,32,0.910256,0.977273,0.877551,0.924731,"[[28, 1], [6, 43]]"
6,Dropout 0.2,"[32, 16, 8]",0.2,0.0010,32,0.897436,0.936170,0.897959,0.916667,"[[26, 3], [5, 44]]"


# Final Experiment Model

In [141]:
final_model = deeper_dropout_03_result["model"]
final_history = deeper_dropout_03_result["history"]

print("Final model: Dropout 0.3")
print("Validation F1:", f"{deeper_dropout_03_result['f1']:.4f}")

Final model: Dropout 0.3
Validation F1: 0.9474


# Final Evaluation

In [142]:
val_probabilities = final_model.predict(
    X_val,
    verbose=0
)

val_predictions = (
    val_probabilities >= 0.5
).astype(int)

final_accuracy = accuracy_score(
    y_val,
    val_predictions
)

final_precision = precision_score(
    y_val,
    val_predictions,
    zero_division=0
)

final_recall = recall_score(
    y_val,
    val_predictions,
    zero_division=0
)

final_f1 = f1_score(
    y_val,
    val_predictions,
    zero_division=0
)

final_cm = confusion_matrix(
    y_val,
    val_predictions
)
TN, FP, FN, TP = final_cm.ravel()

print("final Model — Validation Results")
print(f"Accuracy : {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall   : {final_recall:.4f}")
print(f"F1-score : {final_f1:.4f}")

print("\nConfusion Matrix")
print(final_cm)

print("\nConfusion Matrix Breakdown")
print(f"True Negatives  (TN): {TN}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")
print(f"True Positives  (TP): {TP}")

final Model — Validation Results
Accuracy : 0.9359
Precision: 0.9783
Recall   : 0.9184
F1-score : 0.9474

Confusion Matrix
[[28  1]
 [ 4 45]]

Confusion Matrix Breakdown
True Negatives  (TN): 28
False Positives (FP): 1
False Negatives (FN): 4
True Positives  (TP): 45


In [143]:
test_probabilities = final_model.predict(X_test, verbose=0)

test_predictions = (test_probabilities >= 0.5).astype(int)
test_accuracy = accuracy_score(y_test, test_predictions)

test_precision = precision_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_cm = confusion_matrix(
    y_test,
    test_predictions
)
TN, FP, FN, TP = test_cm.ravel()

print("FINAL TEST RESULTS")
print(f"Accuracy : {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1-score : {test_f1:.4f}")

print("\nConfusion Matrix")
print(test_cm)
print("\nConfusion Matrix Breakdown")
print(f"True Negatives  (TN): {TN}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")
print(f"True Positives  (TP): {TP}")

FINAL TEST RESULTS
Accuracy : 0.8974
Precision: 0.8600
Recall   : 0.9773
F1-score : 0.9149

Confusion Matrix
[[27  7]
 [ 1 43]]

Confusion Matrix Breakdown
True Negatives  (TN): 27
False Positives (FP): 7
False Negatives (FN): 1
True Positives  (TP): 43


In [144]:
final_model.save("final_model.keras")